# 1. Tải llama.cpp và Model từ Hugging Face
Chạy trên Colab Pro (khuyến nghị chọn A100 hoặc L4 GPU để chạy nhanh và mượt nhất)

In [ ]:
!pip install pyngrok huggingface_hub --quiet

# --- THAY ĐỔI MODEL Ở ĐÂY ---
HF_REPO = "AlexHung29629/gemma-4-26B-A4B-it-GGUF"
MODEL_FILE = "gemma-4-26B-A4B-it-Q4_K_M.gguf"
# ----------------------------

# Clone nông (depth 1) và compile siêu tốc CHỈ RIÊNG llama-server cho đúng GPU của Colab (~1-2 phút)
!rm -rf llama.cpp build
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!cmake -B build -S llama.cpp -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build build --config Release --target llama-server -j$(nproc)
print('Build llama-server thành công với CUDA native!')

# Tải model GGUF trực tiếp từ Hugging Face vào thư mục /content/
print(f'Đang tải model {MODEL_FILE} từ Hugging Face...')
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id=HF_REPO, filename=MODEL_FILE, local_dir='/content', local_dir_use_symlinks=False)
print('Tải model thành công!')


# 2. Khởi chạy Llama-server ở Background
Tối ưu hoá các tham số cho throughput trên GPU xịn (A100/L4)

In [ ]:
import os
import subprocess
import glob

LLAMA_ARG_PORT = "8000"
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"  # Fix log streaming

server_path = '/content/build/bin/llama-server'
model_path = f'/content/{MODEL_FILE}'

# Kiểm tra version và CUDA
!{server_path} --version

command = [
    server_path,
    "-m", model_path,
    "--port", LLAMA_ARG_PORT,
    
    # Tham số sinh text
    "--seed", "3407",
    "--temp", "0.6",
    "--top-p", "0.95",
    "--top-k", "20",
    "--min-p", "0.0",
    "--repeat-penalty", "1.0",
    
    # Tối ưu hoá phần cứng GPU
    "--flash-attn", "on",
    "--kv-unified",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    
    # Tối ưu cho Colab Pro (A100 40GB hoặc L4 24GB)
    "--batch-size", "2048",   # Logical batch size
    "--ubatch-size", "512",   # Physical batch size
    "-ngl", "99",             # Offload toàn bộ
    "--ctx-size", "32768",    # Giữ context an toàn 32K
    
    # Server features
    "--parallel", "1",
    "--cont-batching",
    "--jinja",
    "--reasoning", "off",
]

print(f"Đang khởi động llama-server tại {server_path} với model {model_path}...")
llama_process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env=env,
    cwd="/content"
)


# 3. Mở cổng qua Ngrok
Thêm `NGROK_TOKEN` vào tab **Secrets** (Biểu tượng chìa khoá ở thanh công cụ bên trái Colab).

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import time
import sys

try:
    ngrok_token = userdata.get("NGROK_TOKEN")
    !ngrok config add-authtoken {ngrok_token}
except Exception as e:
    print("LỖI: Vui lòng thêm NGROK_TOKEN vào menu Secrets của Colab!")

ngrok.kill()  # Đóng các tunnel ngrok cũ nếu có để tránh lỗi ERR_NGROK_334
public_url = ngrok.connect(LLAMA_ARG_PORT)
print(f"\n👉 COPY ĐỊA CHỈ NÀY DÁN VÀO CẤU HÌNH API: {public_url}\n")

# Stream log của server ra output cell
for line in llama_process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
